# Setting

## Library

In [1]:
# Synthetic Photometry for OSC
# %%
## Library
import os
import glob
import sys
import numpy as np
import shutil
import time
import speclite

start_time = time.time()

from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy import units as u
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.table import hstack
import warnings
warnings.filterwarnings("ignore")

### Plot presetting
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

## 7DT

In [2]:
### Helper Functions
import sys
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *

In [ ]:
# 7DT Setting
sys.path.append('..')
from simulator.helper import *
from simulator.sdtpy import *
register_custom_filters_on_speclite('../simulator')

#	Exposure Time [s]
sdt = SevenDT()
sdt.echo_optics()
filterset = sdt.generate_filterset(bandmin=BANDMIN, bandmax=BANDMAX, bandwidth=BANDWIDTH, bandstep=BANDSTEP, bandrsp=BANDRSP, lammin=LAMMIN, lammax=LAMMAX, lamres=LAMRES)
T_qe = sdt.get_CMOS_IMX455_QE()
sdt.get_optics()
s = sdt.get_sky()
sdt.smooth_sky()
totrsptbl = sdt.calculate_response()
Npix_ptsrc, Narcsec_ptsrc = sdt.get_phot_aperture(exptime=EXPTIME_SINGLE, fwhm_seeing=SEEING, optfactor=EFF_FACTOR, verbose=False)
depthtbl = sdt.get_depth_table(Nsigma=5)
sdt.get_speclite()

# %%
depth_dict = {}
for filtername, depth in zip(depthtbl['name'], depthtbl['5sigma_depth']):
	depth_dict[filtername] = depth

depth_dict

## Function

In [3]:
## Function
def fill_nan_in_array(arr):
    """
    입력된 numpy array에서 nan 값을 양쪽의 유효한 값들을 이용해 채우는 함수.
    
    각 nan 값에 대해:
      - 왼쪽에서 nan이 아닌 값을 찾습니다.
      - 오른쪽에서 nan이 아닌 값을 찾습니다.
      - 두 값 모두 존재하면, 두 값의 중앙값(평균)으로 대체합니다.
      - 한쪽만 존재하면, 그 값을 대체합니다.
      - 양쪽 모두 없으면 그대로 nan으로 둡니다.
    
    Parameters:
      arr (numpy.ndarray): nan이 포함된 1차원 배열
    
    Returns:
      numpy.ndarray: nan이 채워진 새로운 배열
    """
    
    # 원본 배열 복사
    filled_arr = arr.copy()
    n = len(filled_arr)
    
    # 배열의 각 원소를 순회
    for i in range(n):
        if np.isnan(filled_arr[i]):
            left_val = None
            # 왼쪽에서 nan이 아닌 값 찾기
            j = i - 1
            while j >= 0:
                if not np.isnan(filled_arr[j]):
                    left_val = filled_arr[j]
                    break
                j -= 1
            
            right_val = None

            # 오른쪽에서 nan이 아닌 값 찾기
            k = i + 1
            while k < n:
                if not np.isnan(filled_arr[k]):
                    right_val = filled_arr[k]
                    break
                k += 1

            # 대체할 값 결정
            if left_val is not None and right_val is not None:
                filled_arr[i] = np.median([left_val, right_val])
            elif left_val is not None:
                filled_arr[i] = left_val
            elif right_val is not None:
                filled_arr[i] = right_val
            # 양쪽 모두 유효한 값이 없으면 그대로 nan 유지
    return filled_arr

# - Synthetic Photometry
def interpspecfiletrum(lam, flam, filter_lam_min=MIN_7DT_WAVELENGTH, filter_lam_max=MAX_7DT_WAVELENGTH, verbose=False):
	#	Default Flag Type
	flagtype = 'TBD'

	#	Median Resolution
	median_res = np.median(lam.value[1:] - lam.value[:-1])

	#	Wavelength Range
	lammin, lammax = lam.value.min(), lam.value.max()

	#	Padding
	left_pad = np.mean(flam.value[:10])
	right_pad = np.mean(flam.value[-10:])

	#	Interpolation
	##	Spectra has a shorter wavelength range than the filters
	if (lammin > filter_lam_min) & (lammax < filter_lam_max):
		if verbose:
			print(f"Warning! {lammin} < {filter_lam_min} and {lammax} > {filter_lam_max}")

		bluelamarr = np.arange(filter_lam_min-median_res, lammin, median_res)
		redlamarr = np.arange(lammax, filter_lam_max+median_res, median_res)

		interp_lam = np.concatenate((bluelamarr, lam.value, redlamarr))
		interp_flam = np.concatenate((
			np.full(len(bluelamarr), left_pad),
			flam.value,
			np.full(len(redlamarr), right_pad)
		))
		flagtype = 'blue_red'

	##	Spectra has a longer wavelength range than the filters
	elif (lammin > filter_lam_min):
		if verbose:
			print(f"Warning! {lammin} > {filter_lam_min}")
		bluelamarr = np.arange(filter_lam_min-median_res, lammin, median_res)
		# blueflamarr = np.interp(bluelamarr, lam.value, flam.value)
		interp_lam = np.concatenate((bluelamarr, lam.value))
		interp_flam = np.concatenate((np.full(len(bluelamarr), left_pad), flam.value))
		flagtype = 'blue'

	##	Spectra has a longer wavelength range than the filters
	elif (lammax < filter_lam_max):
		if verbose:
			print(f"Warning! {lammax} > {filter_lam_max}")
		redlamarr = np.arange(lammax, filter_lam_max+median_res, median_res)
		interp_lam = np.concatenate((lam.value, redlamarr))
		interp_flam = np.concatenate((flam.value, np.full(len(redlamarr), right_pad)))
		flagtype = 'red'

	##	Spectra is already in the range of filters
	else:
		if verbose:
			print(f"Spectrum is already in the range of filters.")
		interp_lam = lam.value
		interp_flam = flam.value

		flagtype = 'clean'

	return (interp_lam, fill_nan_in_array(interp_flam), flagtype)



### Functions for synthetic photometries 

def process_flux(lam, flam, unit_lam=lamunit, unit_flux=flamunit):
	"""중복 제거 및 NaN 보정 포함한 파장/플럭스 전처리"""
	lam = np.array(lam)
	flam = np.array(flam)

	# 정렬
	sort_idx = np.argsort(lam)
	lam = lam[sort_idx]
	flam = flam[sort_idx]

	# 중복 제거
	_, unique_idx = np.unique(lam, return_index=True)
	lam = lam[unique_idx]
	flam = flam[unique_idx]

	# NaN 보정
	if np.any(np.isnan(flam)):
		print("  - NaN in flux → interpolated")
		flam = fill_nan_in_array(flam)

	return lam * unit_lam, flam * unit_flux


# New function for processing one spectrum to a synthetic photometry FITS table (.tmp)
def process_synphot(nn, specfile, typ, path_save, sdt, fill_nan_in_array, flamunit):
    """
    Process one spectrum to generate a synthetic photometry FITS table (.tmp).
    """
    # Prepare output paths
    basename = os.path.splitext(os.path.basename(specfile))[0]
    table_path = os.path.join(path_save, f"{nn:0>4}.tmp")
    plot_path = os.path.join(path_save, f"{basename}.synphot.png")
    # Skip if already done
    if os.path.exists(table_path):
        return
    # Read spectrum and preprocess flux
    sptbl = Table.read(specfile, format='ascii')
    keys = sptbl.keys()
    lam, flam = process_flux(sptbl[keys[0]], sptbl[keys[1]])
    # Generate synphot observations
    mobstbl = sdt.get_synphot2obs(
        flam, lam, z=None, z0=None,
        figure=(not os.path.exists(plot_path))
    )
    # Save plot if generated
    if not os.path.exists(plot_path):
        plt.title(basename)
        plt.savefig(plot_path)
        plt.close()
    # Build and write output FITS table
    outbl = Table()
    outbl['type'] = [typ]
    outbl['spec'] = [specfile.strip()]
    outbl['flagtype'] = [ 'clean' ]
    for filte in mobstbl['filter']:
        idx = np.where(mobstbl['filter'] == filte)
        for key in mobstbl.keys()[3:]:
            colname = f"{key}_{filte}"
            outbl[colname] = mobstbl[key][idx].item()
            outbl[colname].format = '1.3f'
    outbl.write(table_path, format='fits')

# Data

- KN_spectra_004938_md0.010_vd0.050_mw0.030_vw0.15_ang75.0_z0.009_Av0.000_phase1.0.fits

In [12]:
meta_table_path = os.path.join(SPECTRA_WOLLAEGER_DATA, "d40Mpc/param.csv")
metatbl = Table.read(meta_table_path)
print(f"Path: {meta_table_path}")
print(f"{len(metatbl)} rows")
metatbl[:3]

Path: /home/gpaek/SED-Classifier/notebook/../data/Spectra/Wollaeger+21/d40Mpc/param.csv
11025 rows


col0,md,vd,mw,vw,ang,phase,z,Av
int64,float64,float64,float64,float64,int64,float64,float64,float64
0,0.001,0.05,0.001,0.05,0,0.125,0.009183069776131852,0.0
1,0.001,0.05,0.001,0.05,0,0.25,0.009183069776131852,0.0
2,0.001,0.05,0.001,0.05,0,0.5,0.009183069776131852,0.0


In [16]:
mdarr = metatbl["md"].value
vdarr = metatbl["vd"].value
mwarr = metatbl["mw"].value
vwarr = metatbl["vw"].value
angarr = metatbl["ang"].value
phasearr = metatbl["phase"].value
zarr = metatbl["z"].value
Avarr = metatbl["Av"].value
#
unique_mdarr = np.unique(mdarr)
unique_vdarr = np.unique(vdarr)
unique_mwarr = np.unique(mwarr)
unique_vwarr = np.unique(vwarr)
unique_angarr = np.unique(angarr)
unique_phasearr = np.unique(phasearr)
unique_zarr = np.unique(zarr)
unique_Avarr = np.unique(Avarr)
#
print(f"md:    {np.unique(mdarr)}")
print(f"vd:    {np.unique(vdarr)}")
print(f"mw:    {np.unique(mwarr)}")
print(f"vw:    {np.unique(vwarr)}")
print(f"ang:   {np.unique(angarr)}")
print(f"phase: {np.unique(phasearr)}")
print(f"z:     {np.unique(zarr)}")
print(f"Av:    {np.unique(Avarr)}")
#
n_combinations = len(unique_mdarr) * len(unique_vdarr) * len(unique_mwarr) * len(unique_vwarr) * len(unique_angarr) * len(unique_phasearr) * len(unique_zarr) * len(unique_Avarr)

print(f"Number of Combinations: {n_combinations}")

md:    [0.001 0.003 0.01  0.03  0.1  ]
vd:    [0.05 0.15 0.3 ]
mw:    [0.001 0.003 0.01  0.03  0.1  ]
vw:    [0.05 0.15 0.3 ]
ang:   [ 0 15 30 45 60 75 90]
phase: [0.125 0.25  0.5   1.    1.5   2.    3.   ]
z:     [0.00918307]
Av:    [0.]
Number of Combinations: 11025


In [30]:
for nn, (md, vd, mw, vw, ang, phase, z, Av) in enumerate(zip(mdarr, vdarr, mwarr, vwarr, angarr, phasearr, zarr, Avarr)):

    # filename = f"KN_spectra_{}"
    filename_pattern = f"{SPECTRA_WOLLAEGER_DATA}/d40Mpc/KN_spectra_*_md{md:.3f}_vd{vd:.3f}_mw{mw:.3f}_vw{vw}_ang{ang:.1f}_z{z:.3f}_Av{Av:.3f}_phase{phase}.fits"

    # print(nn, (md, vd, mw, vw, ang, phase, z, Av))
    n = len(glob.glob(filename_pattern))
    # print(os.path.basename(filename_pattern))

    if n == 0:
        print(nn)

10000
10001
10002
10003
10004
10005
10006
10007
10008
10009
10010
10011
10012
10013
10014
10015
10016
10017
10018
10019
10020
10021
10022
10023
10024
10025
10026
10027
10028
10029
10030
10031
10032
10033
10034
10035
10036
10037
10038
10039
10040
10041
10042
10043
10044
10045
10046
10047
10048
10049
10050
10051
10052
10053
10054
10055
10056
10057
10058
10059
10060
10061
10062
10063
10064
10065
10066
10067
10068
10069
10070
10071
10072
10073
10074
10075
10076
10077
10078
10079
10080
10081
10082
10083
10084
10085
10086
10087
10088
10089
10090
10091
10092
10093
10094
10095
10096
10097
10098
10099
10100
10101
10102
10103
10104
10105
10106
10107
10108
10109
10110
10111
10112
10113
10114
10115
10116
10117
10118
10119
10120
10121
10122
10123
10124
10125
10126
10127
10128
10129
10130
10131
10132
10133
10134
10135
10136
10137
10138
10139
10140
10141
10142
10143
10144
10145
10146
10147
10148
10149
10150
10151
10152
10153
10154
10155
10156
10157
10158
10159
10160
10161
10162
10163
10164
10165
1016

: 